# Gemma paraphrase pipeline for processed news parquet files

This notebook reads parquet files from Google Drive, paraphrases each text segment with Gemma, and saves one output parquet per input parquet back to Drive. It is resumable at the file level: if an output file already exists, that input file is skipped.

In [ ]:
# Install dependencies
!pip -q install -U transformers accelerate sentencepiece pandas pyarrow huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/content/drive/MyDrive/news_data/pre_2022_processed")
OUTPUT_DIR = Path("/content/drive/MyDrive/news_data/pre_2022_gemma2_2b_paraphrased")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_CACHE_DIR = Path("/content/gemma_paraphrase_cache")
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/gemma-2-2b-it"

BATCH_SIZE = 16
MAX_INPUT_TOKENS = 384
MAX_NEW_TOKENS = 320
TEMPERATURE = 0.7
TOP_P = 0.9
DO_SAMPLE = True
REPETITION_PENALTY = 1.05

MAX_FILES = None
OVERWRITE = False

TEXT_COLUMN_CANDIDATES = ["chunk_text", "text", "maintext"]
WRITE_SUMMARY_JSON = True


In [ ]:
# Optional Hugging Face login if needed
from huggingface_hub import login
login()


In [ ]:
import gc
import json
import os
import shutil

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    "device_map": "auto" if torch.cuda.is_available() else None,
}
try:
    model_kwargs["attn_implementation"] = "sdpa"
except Exception:
    pass

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.eval()

print("Loaded:", MODEL_NAME)


In [ ]:
SYSTEM_PROMPT = (
    "You are rewriting text for a controlled evaluation dataset. "
    "Paraphrase the user's text into natural, fluent English while preserving the original meaning, tone, register, and approximate length. "
    "Do not add facts, commentary, headings, bullet points, or explanations. "
    "Output only the rewritten text."
)

def choose_text_column(df: pd.DataFrame) -> str:
    for col in TEXT_COLUMN_CANDIDATES:
        if col in df.columns:
            return col
    raise ValueError(f"No text column found. Tried: {TEXT_COLUMN_CANDIDATES}")

def clean_text_minimal(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\xa0", " ")
    text = " ".join(text.split())
    return text.strip()

def build_chat_prompt(text: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return f"Instruction: {SYSTEM_PROMPT}\n\nText:\n{text}\n\nRewrite:"

def generate_batch(texts):
    prompts = [build_chat_prompt(t) for t in texts]
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )
    if torch.cuda.is_available():
        enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    input_lengths = enc["attention_mask"].sum(dim=1).tolist()
    generations = []
    for out_ids, in_len in zip(outputs, input_lengths):
        gen_ids = out_ids[in_len:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        generations.append(text)

    del enc, outputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return generations

def adaptive_generate(texts, initial_batch_size=BATCH_SIZE):
    results = []
    start = 0
    current_bs = initial_batch_size
    while start < len(texts):
        batch = texts[start:start+current_bs]
        try:
            batch_out = generate_batch(batch)
            results.extend(batch_out)
            start += len(batch)
        except torch.cuda.OutOfMemoryError:
            if current_bs == 1:
                raise
            current_bs = max(1, current_bs // 2)
            print(f"OOM hit; reducing batch size to {current_bs}")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return results


In [ ]:
def process_one_file(input_path: Path, output_dir: Path):
    print(f"\n=== Processing {input_path.name} ===")
    local_in = LOCAL_CACHE_DIR / input_path.name
    shutil.copy2(input_path, local_in)

    df = pd.read_parquet(local_in)
    rows_loaded = len(df)
    print("Loaded rows:", rows_loaded)

    text_col = choose_text_column(df)
    df = df.copy()
    df[text_col] = df[text_col].astype(str).map(clean_text_minimal)
    df = df[df[text_col].ne("")].reset_index(drop=True)
    print("Rows after preprocessing:", len(df))

    original_texts = df[text_col].tolist()
    generated_texts = adaptive_generate(original_texts, initial_batch_size=BATCH_SIZE)

    out_df = df.copy()
    out_df["original_text"] = out_df[text_col]
    out_df["ai_generated_text"] = generated_texts
    out_df["generator_model"] = MODEL_NAME
    out_df["generator_temperature"] = TEMPERATURE
    out_df["generator_top_p"] = TOP_P
    out_df["generator_max_new_tokens"] = MAX_NEW_TOKENS

    local_out = LOCAL_CACHE_DIR / f"{input_path.stem}_gemma2_2b_paraphrased.parquet"
    out_df.to_parquet(local_out, index=False)

    drive_out = output_dir / local_out.name
    shutil.copy2(local_out, drive_out)
    print("Saved parquet to:", drive_out)

    summary = {
        "input_file": input_path.name,
        "output_file": drive_out.name,
        "rows_loaded": int(rows_loaded),
        "rows_output": int(len(out_df)),
        "text_column": text_col,
        "model_name": MODEL_NAME,
        "batch_size_used": BATCH_SIZE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
    }

    if WRITE_SUMMARY_JSON:
        local_json = LOCAL_CACHE_DIR / f"{input_path.stem}_gemma_paraphrased_summary.json"
        with open(local_json, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        shutil.copy2(local_json, output_dir / local_json.name)

    try:
        local_in.unlink(missing_ok=True)
        local_out.unlink(missing_ok=True)
    except Exception:
        pass

    return summary


In [ ]:
input_files = sorted(INPUT_DIR.glob("*.parquet"))
if MAX_FILES is not None:
    input_files = input_files[:MAX_FILES]

print(f"Found {len(input_files)} parquet files in {INPUT_DIR}")

summaries = []
for input_path in input_files:
    output_path = OUTPUT_DIR / f"{input_path.stem}_gemma2_2b_paraphrased.parquet"
    if output_path.exists() and not OVERWRITE:
        print(f"Skipping {input_path.name} (already paraphrased)")
        continue

    summary = process_one_file(input_path, OUTPUT_DIR)
    summaries.append(summary)

print("\nRun complete.")
print(f"Newly processed files this run: {len(summaries)}")


In [ ]:
if summaries:
    summary_df = pd.DataFrame(summaries)
    summary_csv = OUTPUT_DIR / "run_summary.csv"
    if summary_csv.exists():
        try:
            old = pd.read_csv(summary_csv)
            summary_df = pd.concat([old, summary_df], ignore_index=True)
            summary_df = summary_df.drop_duplicates(subset=["input_file", "output_file"], keep="last")
        except Exception:
            pass
    summary_df.to_csv(summary_csv, index=False)
    print("Saved run summary to:", summary_csv)
else:
    print("No new files processed in this run.")


In [ ]:
out_files = sorted(OUTPUT_DIR.glob("*_gemma2_2b_paraphrased.parquet"))
print("Output files:", len(out_files))
if out_files:
    sample = pd.read_parquet(out_files[0])
    display(sample.head(3))
    print(sample.columns.tolist())
